<a href="https://colab.research.google.com/github/EenPutra/Tugas-3-Kecerdasan-Buatan-Een-H-P-6022251023/blob/main/Task_3_AI_Een_6022251023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# MEMBERSHIP FUNCTIONS
# =========================
def trapmf(x, a, b, c, d):
    return np.maximum(np.minimum(np.minimum((x-a)/(b-a+1e-6), 1), (d-x)/(d-c+1e-6)), 0)

def trimf(x, a, b, c):
    return np.maximum(np.minimum((x-a)/(b-a+1e-6), (c-x)/(c-b+1e-6)), 0)

# =========================
# RANGE
# =========================
area_range = np.linspace(0, 1000, 1000)
power_range = np.linspace(0, 500, 1000)

# =========================
# MEMBERSHIP INPUT
# =========================
area_sedikit = trapmf(area_range, 0, 0, 200, 400)
area_sedang  = trimf(area_range, 300, 500, 700)
area_banyak  = trapmf(area_range, 600, 800, 1000, 1000)

power_kecil    = trapmf(power_range, 0, 0, 100, 200)
power_menengah = trimf(power_range, 150, 250, 350)
power_besar    = trapmf(power_range, 300, 400, 500, 500)

# =========================
# PLOT INPUT MEMBERSHIP
# =========================
plt.figure(figsize=(6,5))

#plt.subplot(1,2,1)
plt.plot(area_range, area_sedikit, label='Sedikit')
plt.plot(area_range, area_sedang, label='Sedang')
plt.plot(area_range, area_banyak, label='Banyak')
plt.title("Membership Area PV")
plt.xlabel("Area (m2)")
plt.ylabel("Membership")
plt.legend()
plt.grid()

plt.show()
plt.figure(figsize=(6,5))

#plt.subplot(1,2,2)
plt.plot(power_range, power_kecil, label='Kecil')
plt.plot(power_range, power_menengah, label='Menengah')
plt.plot(power_range, power_besar, label='Besar')
plt.title("Membership Daya PV")
plt.xlabel("Daya (kW)")
plt.ylabel("Membership")
plt.legend()
plt.grid()

plt.show()

# =========================
# OUTPUT MEMBERSHIP (SINGLETON)
# =========================
z_output = np.array([1, 2, 3])  # Basic, Premium, Platinum
labels = ['Basic', 'Premium', 'Platinum']

plt.figure(figsize=(6,4))
plt.scatter(z_output, [1,1,1])
plt.title("Output Membership (Singleton Sugeno)")
plt.xlabel("Output Value")
plt.ylabel("Membership = 1 (Singleton)")
plt.xticks(z_output, labels)
plt.grid()
plt.show()



In [ ]:
# =========================
# INPUT USER
# =========================
A = float(input("Masukkan Luas Area PV (0-1000 m2): "))
P = float(input("Masukkan Daya PV (0-500 kW): "))

# =========================
# INTERPOLASI MEMBERSHIP
# =========================
def interp_membership(x_range, mf, val):
    return np.interp(val, x_range, mf)

μ_A = {
    "sedikit": interp_membership(area_range, area_sedikit, A),
    "sedang":  interp_membership(area_range, area_sedang, A),
    "banyak":  interp_membership(area_range, area_banyak, A)
}

μ_P = {
    "kecil":    interp_membership(power_range, power_kecil, P),
    "menengah": interp_membership(power_range, power_menengah, P),
    "besar":    interp_membership(power_range, power_besar, P)
}

print("\n=== DERajat KEANGGOTAAN ===")
print("Area:", μ_A)
print("Daya:", μ_P)

# =========================
# RULE BASE
# =========================
rules = [
    ("sedikit","kecil",1),
    ("sedikit","menengah",1),
    ("sedikit","besar",2),
    ("sedang","kecil",1),
    ("sedang","menengah",2),
    ("sedang","besar",3),
    ("banyak","kecil",2),
    ("banyak","menengah",3),
    ("banyak","besar",3),
]

# =========================
# INFERENSI
# =========================
weights = []
z_values = []

print("\n=== RULE EVALUATION ===")

for i, (area, power, z) in enumerate(rules):
    w = μ_A[area] * μ_P[power]
    weights.append(w)
    z_values.append(z)
    print(f"Rule {i+1}: w = {w:.4f}, output = {z}")

weights = np.array(weights)
z_values = np.array(z_values)

# =========================
# DEFUZZIFIKASI
# =========================
if np.sum(weights) == 0:
    z_final = 0
else:
    z_final = np.sum(weights * z_values) / np.sum(weights)

# =========================
# INTERPRETASI OUTPUT
# =========================
if z_final <= 1.5:
    kategori = "Basic"
elif z_final <= 2.5:
    kategori = "Premium"
else:
    kategori = "Platinum"

print("\n=== HASIL AKHIR ===")
print(f"Nilai Defuzzifikasi: {z_final:.3f}")
print(f"Paket Terpilih: {kategori}")

# =========================
# PLOT OUTPUT (SUGENO)
# =========================
plt.figure(figsize=(10,5))

plt.stem(z_values, weights, basefmt=" ")

plt.axvline(z_final, linestyle='--', linewidth=2)

plt.title("Sugeno Output (Firing Strength vs Singleton)")
plt.xlabel("Output (1=Basic, 2=Premium, 3=Platinum)")
plt.ylabel("Firing Strength")

plt.xticks([1,2,3], ['Basic','Premium','Platinum'])
plt.grid()

plt.show()